# 49655447 — what are these networks actually fitting?

Eval-only, no training. Finished 2026-10-26, ExitCode `0:0`, elapsed `00:18:42`.

**Headline: they were not solving graph colouring.** Permuting each node's neighbour list — same
edge set, same colouring, same hints, same DD trace, all asserted equal — drops every checkpoint
tested from 0.94–1.00 to 0.00–0.15 at p < 1e-100. The planted colours were partly readable off
token position, because `planted_regular_graph` appends edges in class-pair block order and
`encode_instance` emits `adj[v]` in that raw order.

## Provenance

| | |
|---|---|
| package commit | `e25dcf3` (the run). The fix, `5103e83`, came after and does not affect these numbers. |
| seed | `0` for every checkpoint scored; `--seed 0` for the permutations |
| held-out instances | `seed_start=50000000000`, instance *i* at `seed_start+i` |
| part A | n=600 per side (own training prefix vs held-out) |
| part B | n=400, four orderings, paired by instance index |
| deck | `notebooks/sudoku_leak_2026-10-26/encoding_leak.pdf` (6 pages, sent to OneDrive) |

### Configs

All share degree=5, n_colors=4, rho=0.3, rho_max=0.95, removal hints, d_model=256, nhead=4,
pre-norm, lr=1e-3 constant, batch=32.

| role | run dir | L | ff | n_train | epochs |
|---|---|---|---|---|---|
| part A, N=48 | `sudoku_layer_scan_2026-10-24_n48/d0256_h04_L12_ff01024_lr1e-03_pre` | 12 | 1024 | 20000 | 150 |
| part A, N=96 ctrl | `sudoku_n96_baseline_2026-10-25/d0256_h04_L12_ff01024_lr1e-03_pre` | 12 | 1024 | 20000 | 150 |
| part A+B, N=96 cell B | `sudoku_n96_plateau_2026-10-26/d0256_h04_L12_ff01024_lr1e-03_pre` | 12 | 1024 | 80000 | 150 |
| part B, N=24 record | `$OAK/.../sudoku_dd_..._sharedpool0.9960_job48912548` | 12 | 1024 | 80000 | 300 |
| part B, N=48 best | `sudoku_ff_depth_tradeoff_2026-10-25_n48/d0256_h04_L08_ff00256_lr1e-03_pre` | 8 | 256 | 20000 | 150 |

## Comparison to the previous best on the same eval

**There is none, for either quantity.** No run in this project had ever been scored on its own
training instances, and none had ever been scored under a permuted adjacency. What these numbers
displace is the solve_rate ladder itself (0.9960 at N=24, 0.9573 at N=48, 0.9273 at N=96), which
they reinterpret rather than beat: those scores are real on this generator, but they measure the
depth and width needed to decode a positional code, not to execute T rounds of deduction.

In [ ]:
import csv, os
import matplotlib.pyplot as plt
import numpy as np

SRC = os.environ["SCRATCH"] + "/spin_gnn/sudoku_memorization_2026-10-26"
load = lambda f: list(csv.DictReader(open(os.path.join(SRC, f))))
ordr, memr = load("ordering_counterfactual.csv"), load("memorization_gap.csv")
print(f"{len(ordr)} ordering rows, {len(memr)} memorization rows")

## Part B — the ordering counterfactual (the result)

`within_block` permutes only among neighbours of the **same** colour, so the colour code survives
and only the order changes. Within-block order is already arbitrary at chance in training (sorted
by node id in 38.9% of runs at N=48, 36.2% at N=96), so it is **in-distribution** and is the
control that separates "the order changed" from "the colour code was destroyed". It ties identity
on all three models, which is what licenses reading the collapse as the colour code.

In [ ]:
ORDER = ["identity", "within_block", "shuffle", "byid"]
LABEL = {(24, 12, 1024): "N=24 record (0.9960)", (96, 12, 1024): "N=96 cell B (0.9273)",
         (48, 8, 256): "N=48 best (0.9573)"}

byk = {}
for r in ordr:
    byk.setdefault((int(r["n_nodes"]), int(r["num_layers"]), int(r["dim_feedforward"])),
                   {})[r["ordering"]] = r

print(f"{'model':24s} {'ordering':14s} {'solve':>7s} {'delta':>9s} {'paired p':>11s} {'wf':>6s}")
for k, byo in byk.items():
    for o in ORDER:
        r = byo[o]
        d = f"{float(r['gap']):+.4f}" if r["gap"] else ""
        p = f"{float(r['p']):.2e}" if r["p"] else ""
        print(f"{LABEL[k] if o == 'identity' else '':24s} {o:14s} "
              f"{float(r['heldout_solve_rate']):7.4f} {d:>9s} {p:>11s} "
              f"{float(r['well_formed_rate']):6.3f}")
    print()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
xs, w = np.arange(len(ORDER)), 0.26
for i, (k, byo) in enumerate(byk.items()):
    v = [float(byo[o]["heldout_solve_rate"]) for o in ORDER]
    lo = [v[j] - float(byo[o]["heldout_wilson_lo"]) for j, o in enumerate(ORDER)]
    hi = [float(byo[o]["heldout_wilson_hi"]) - v[j] for j, o in enumerate(ORDER)]
    ax.bar(xs + (i - 1) * w, v, w, yerr=[lo, hi], capsize=3, label=LABEL[k],
           edgecolor="black", linewidth=0.7)
ax.set_xticks(xs); ax.set_xticklabels(ORDER)
ax.set_ylabel("solve_rate (n=400 held-out)"); ax.set_ylim(0, 1.15)
ax.axhline(0.90, color="0.5", ls="--", lw=1.2)
ax.set_title("Same problems, different neighbour order")
ax.legend(fontsize=9); fig.tight_layout()

## Part A — memorization, measured directly

Training index *i* is reproducible as `build_instances(cfg, 1, i)`, so a prefix of the training set
is recoverable exactly and the model can be scored on the instances it actually saw. The training
set is built **once**, before the epoch loop (`train_sudoku_transformer.py:345` vs `:382`), so a
150-epoch run sees the same instances 150 times and *can* memorize them.

Unpaired two-proportion z-test: the train sample and the held-out sample are different instances,
so McNemar does not apply here (it does apply to part B, where the instances are shared).

In [ ]:
print(f"{'run':22s} {'train':>7s} {'held-out':>9s} {'gap':>9s} {'p':>10s}")
for r in memr:
    nt = r["n_train_instances"] or "20000"
    print(f"N={r['n_nodes']:>2s} L={r['num_layers']:>2s} @{int(nt):<6d}".ljust(22) +
          f" {float(r['train_solve_rate']):7.4f} {float(r['heldout_solve_rate']):9.4f}"
          f" {float(r['gap']):+9.4f} {float(r['p']):10.3g}")

Positive and resolvable at 20000 on both sizes, absent at 80000 — **data closes the memorization
gap**, pre-registered in advance and confirmed.

### What this retired

The project had been inferring memorization from the train−val NLL gap in `history.json`. That
diagnostic does not work: `train_nll` is `epoch_loss / epoch_n`, averaged over the epoch **while
the weights update**, while `val_nll` uses end-of-epoch weights. Measured bias on the N=96
control: **+0.3263 at epoch 10**, decaying to −0.0289 by the final epoch as the curve flattens.
Against `n_val=300`, whose own difficulty draw is worth a few hundredths, the *sign* of a ±0.03 gap
carries no information.

Finding 15's **mechanism** — "cell B is the only N=96 run with a positive train−val gap (+0.0239),
so it fits and then generalizes" — rested on exactly that and is withdrawn. Its **result**, data as
the binding lever at +0.1007 on 1500 pool instances (p=3e-20), is unaffected.

## Pre-registrations, as resolved

| | prediction | outcome |
|---|---|---|
| (a) | gap positive at 20000, smaller at 80000 | **confirmed** (+0.0583, +0.0717, −0.0133 p=0.355) |
| (b) | `within_block` ties `identity` (design check) | **confirmed** on all three (p = 1, 0.175, 1) |
| (c) | `shuffle` resolvably worse everywhere | **confirmed**, p < 1e-100 on all three |
| (d) | the drop is larger at N=96 than N=24 | **uninformative** — all three floor out, and N=48's drop (−0.9300) exceeds N=96's (−0.9025), so there is no hardness gradient to read. Flagged as a guess in advance. |

## The fix, and what it costs

`SudokuReasoningConfig.shuffle_adj` permutes `adj[v]` in `build_instances` from a dedicated rng
stream (the 40M offset, following the existing 10/20/30M convention) — not inside
`planted_regular_graph`, because five other scripts call that function for the edge set alone and
would have their streams perturbed for nothing. Verified: colour-sorted 100.0% → 5.7% at N=24 and
9.1% at N=48 (chance for d=5 over q=4), the pre-fix generator stream stays bit-identical, and the
edge set, colouring, hints and DD trace are unchanged instance by instance.

`shuffle_adj` is in `sudoku_agg_eval.INSTANCE_KEYS`, so `check_comparable` **refuses** to pool a
`_shuf` cell with any earlier checkpoint. That is deliberate — they are different instance
distributions — but it means re-measuring a result requires retraining it. `49659502` retrains the
record config with one change (`--shuffle-adj`) to establish the clean baseline; which of findings
11/13/14/15 to redo after that is a scoping decision, not an automatic one.